In [ ]:
import re
import json
import random

from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset , DataLoader
from datasets import load_dataset

import matplotlib.pyplot as plt

random.seed(42)
torch.manual_seed(42)

device = torch.device("cpu")

print("Training device: ",device)

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

train_subset = dataset["train"].shuffle(seed=42)

split_data = train_subset.train_test_split(
    test_size=0.2,
    seed=42
)

train_data = split_data["train"]
val_data = split_data["test"]

test_data = dataset["test"]

print("Training reviews: ",len(train_data))
print("Validation reviews: ",len(val_data))
print("Test reviews: ",len(test_data))

print(train_data[0]["text"][:5000])
print("Label: ",train_data[0]["label"])

MAX_VOCAB = 20000
MAX_LENGTH = 200

def tokenize(text):
    return re.findall(r"[a-z0-9']+",text.lower())

word_counts = Counter()

for item in train_data:
    word_counts.update(tokenize(item["text"]))

vocab = {
    "<PAD>":0,
    "<UNK>":1
}

for word,count in word_counts.most_common(MAX_VOCAB - 2):
    if count >= 2:
        vocab[word] = len(vocab)
print("Vocabulary size: ",len(vocab))

def encode_text(text):
    tokens = tokenize(text)

    ids = [
    vocab.get(token,vocab["<UNK>"])
    for token in tokens
    ]

    ids = ids[:MAX_LENGTH]

    padding_needed = MAX_LENGTH - len(ids)

    ids += [vocab["<PAD>"]] * padding_needed

    return ids

class ReviewDataset(Dataset):
    def __init__(self,records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self,index):
        text = self.records[index]["text"]
        label = self.records[index]["label"]

        input_ids = torch.tensor(
            encode_text(text),
            dtype=torch.long
        )

        target = torch.tensor(
            label,
            dtype=torch.float32
        )
        return input_ids , target

train_loader = DataLoader(
    ReviewDataset(train_data),
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    ReviewDataset(val_data),
    batch_size=32,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    ReviewDataset(test_data),
    batch_size=32,
    shuffle=False,
    num_workers=0
)

class SentimentModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=64
    ):
        super().__init__()

        self.embedding = nn.Embedding(
        vocab_size,
        embedding_dim,
        padding_idx=0
        )

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(
        embedding_dim,
        1
        )
    def forward(self,input_ids):
        embedded = self.embedding(input_ids)

        mask = (input_ids != 0).unsqueeze(-1)

        masked_embeddings = embedded * mask

        summed_embeddings = masked_embeddings.sum(dim=1)

        token_counts = mask.sum(dim=1).clamp(min=1)

        mean_embeddings = summed_embeddings/token_counts

        mean_embeddings = self.dropout(mean_embeddings)

        logits = self.classifier(mean_embeddings)

        return logits.squeeze(1)

model = SentimentModel(
    vocab_size=len(vocab),
    embedding_dim=64
).to(device)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

def train_one_epoch(model,loader,loss_fn,optimizer):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for input_ids,labels in loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = loss_fn(logits,labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size

        predictions = (logits>=0).float()

        correct += (predictions == labels).sum().item()
        total += batch_size

    average_loss = total_loss/total

    accuracy = correct/total

    return average_loss , accuracy

def evaluate(model, loader, loss_fn):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for input_ids, labels in loader:

            input_ids = input_ids.to(device)
            labels = labels.to(device)

            logits = model(input_ids)

            loss = loss_fn(logits, labels)

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size

            predictions = (logits >= 0).float()

            correct += (predictions == labels).sum().item()

            total += batch_size

    return total_loss / total, correct / total

EPOCHS = 12

history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": []
}

best_val_loss = float("inf")
best_state = None

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, loss_fn, optimizer
    )

    val_loss, val_acc = evaluate(
        model, val_loader, loss_fn
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_accuracy"].append(train_acc)
    history["val_accuracy"].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

model.load_state_dict(best_state)

epochs_range = range(1,len(history["train_loss"])+1)

plt.subplot(1,2,1)

plt.plot(
    epochs_range,
    history["train_loss"],
    marker='o',
    label="Training Loss"
)

plt.plot(
    epochs_range,
    history["val_loss"],
    marker='o',
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Vs Validation Loss")

plt.legend()

plt.tight_layout()
plt.show()

test_loss , test_accuracy = evaluate(
    model,
    test_loader,
    loss_fn
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")